# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan-dev/flyrank-ml-internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's (content_hash_id) aggregated behavioral profile for March 2026 (report_date from 2026-03-01 to 2026-03-31), built by rolling up daily rows from fact_content_daily_performance to the content_hash_id grain. Each content item's monthly profile is the unit that gets clustered into an archetype — not the raw daily rows themselves, and not a multi-month rolling window.

In [18]:
import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [19]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

df_files = con.sql("""
SELECT *
FROM glob('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**')
""").df()

for f in df_files["file"]:
    print(f)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-09/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance

In [20]:
con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [21]:
con.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT content_hash_id) AS n_content_items,
  COUNT(DISTINCT client_hash_id) AS n_clients,
  MIN(report_date) AS min_date,
  MAX(report_date) AS max_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

,total_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [22]:
con.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
  SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** — knowable at the decision moment, safe to cluster on:
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (aggregated to monthly per item, only where `gsc_data_available IS TRUE`)
- `ga4_pageviews`, `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec` (only where `ga4_data_available IS TRUE`)
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai` (traffic-source mix — useful for archetype shape)
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` (AI-referral breakdown, potentially collapsed into a single `pct_ai_sessions` ratio feature)
- `content_type` from `dim_content` (static attribute, not derived from an outcome)

**Label / proxy** — not applicable for real clustering (this is unsupervised), but reserved for the leakage-trap exercise only:
- No true label column exists in `fact_content_daily_performance` or (pending your check) `dim_content`. For the trap, I'll construct a proxy — e.g., comparing `avg_gsc_position` in March vs. a later month to derive a synthetic "improving/declining" tag — used only to demonstrate leakage, never as a real input.

**Context** — for grouping, joining, filtering; never fed to the model:
- `content_hash_id` — the clustering unit itself, used to group rows, never as a feature value.
- `client_hash_id` — used only for join/filter/grouped-split purposes (e.g., checking client history depth), never as a feature.
- `report_date` — used to filter to the `2026-03` window, not passed to the clustering algorithm.
- `gsc_data_available`, `ga4_data_available`, `client_has_gsc`, `client_has_ga4` — availability flags, used to filter/gate which rows are safe to aggregate, not clustering inputs themselves.
- `month` — partition key, used for filtering only.

**Excluded** — each with a one-line why:
- `gsc_sum_position` — redundant with `gsc_avg_position`; including both would let one dominate variance for no added signal.
- Any GA4-derived rate that mixes `ga4_data_available = FALSE` rows — these are zero-filled placeholders, not real zero-engagement content; including them would fabricate a false "low engagement" archetype.
- Rows from clients with very short history relative to `2025-01` (checked via `dim_clients.gsc_data_start`) — excluded from the feature build for that client's earliest days, since absence there isn't the same as zero activity.
- `scroll_events` — excluded from the initial five-feature set only because its measurement scale/denominator isn't documented here; flagged to check against `docs/data-dictionary.md` before adding it, rather than guessing at what "high" vs "low" means.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [24]:
con.sql("""
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT content_hash_id) AS n_content_items,
       COUNT(DISTINCT client_hash_id) AS n_clients, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

,total_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [25]:
con.sql("""
SELECT COUNT(*) AS total_rows,
       SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


In [29]:
con.sql("""
SELECT
  ga4_data_available,
  COUNT(*) AS rows,
  SUM(CASE WHEN sessions_organic IS NULL THEN 1 ELSE 0 END) AS null_sessions_organic,
  SUM(CASE WHEN sessions_ai IS NULL THEN 1 ELSE 0 END) AS null_sessions_ai
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY ga4_data_available
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows,null_sessions_organic,null_sessions_ai
0,<NA>,3018741,3018741.0,3018741.0
1,False,6408671,0.0,0.0
2,True,413966,0.0,0.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Based on what you've actually verified in queries 1–3, here's a limitations section grounded in real numbers rather than guesses:

**GA4 coverage is extremely thin — 4.2% of rows.** Only 413,966 of 9,841,378 March rows have `ga4_data_available IS TRUE`. This means any engagement, session-source, or AI-referral feature is really only observable for a small, non-random slice of content items — likely the clients who adopted GA4 tracking, not a representative cross-section. Clustering on these features risks producing an archetype that's really just "has GA4 tracking" vs. "doesn't," rather than a genuine behavioral pattern.

**GSC coverage is also partial — 36.7% of rows.** Even the more available signal (`gsc_data_available`) covers only about a third of rows, meaning roughly two-thirds of daily records this month carry no reliable search-performance data at all for that content item that day. Archetypes built on GSC aggregates will systematically underrepresent items or days with thin search visibility.

**Client history depth is uneven and unverified in this pass.** I attempted to check `dim_clients.gsc_data_start`/`ga4_data_start` to see how much real history each of the 55 clients has, but couldn't resolve the correct file path for `dim_clients` in this session (a 404 on the expected `data_0.parquet` naming, unlike the fact table). So this limitation is stated on the strength of the skill notes and the GSC/GA4 availability skew observed above, not a query I was able to execute and verify myself — that's a real gap in this contract, not a rounding error, and it should be closed before relying on the claim.

**A single month can't distinguish a stable pattern from a one-month blip.** Archetypes are meant to describe durable behavioral tendencies, but this contract only aggregates March 2026. A content item having an unusual month (a traffic spike, an outage, a one-off campaign) will look identical in this data to a genuinely different long-run archetype.

**Zero-filled placeholders vs. true zeros.** Per the `flyrank-data` skill, rows before a client's GA4 onboarding date are zero-filled with `ga4_data_available = FALSE`, not simply absent — so any feature built without checking that flag risks treating "no tracking yet" as "no engagement," which is a different fact entirely.

I'd flag the second point above (unresolved `dim_clients` path) honestly in your notebook rather than smoothing over it — it's a real unresolved verification gap, and part of the exercise is showing what you could and couldn't confirm. Want to go back and get that path resolved before finishing section 4, or note it as-is and move to section 5 (self-check) with this as an open item?

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.